# Aviation Crisis Management & Financial Simulation 

**Objective:** To simulate the financial impact of flight cancellations and diversions using Big Data (5.8M records), transforming operational anomalies into business insights.

In [ ]:
# Importing Libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

## 1- Loading & Merging Data

In [ ]:
# Loading 
flights_df=pd.read_csv(r'Airlines_db\flights.csv')
airlines_df=pd.read_csv(r'Airlines_db\airlines.csv')
airports_df=pd.read_csv(r'Airlines_db\airports.csv')
print(f'Flight shape : {flights_df.shape}\nAirlines shape : {airlines_df.shape}\nAirports shape: {airports_df.shape}')

In [ ]:
# standardize column names
flights_df.columns=[c.strip().upper() for c in flights_df.columns]
airlines_df.columns=[c.strip().upper() for c in airlines_df.columns]
airports_df.columns=[c.strip().upper() for c in airports_df]

In [ ]:
# 1-Merging  flights with Airlines
airline_lkp=airlines_df.rename(columns={'IATA_CODE': 'AIRLINE', 'AIRLINE': 'AIRLINE_NAME'})
df_merged = flights_df.merge(airline_lkp[['AIRLINE', 'AIRLINE_NAME']],on="AIRLINE",how='left')
df_merged.head()

In [ ]:
origin_lkp =airports_df.rename(columns={
    'IATA_CODE': 'ORIGIN_AIRPORT', 'AIRPORT': 'ORIGIN_AIRPORT_NAME', 'CITY': 'ORIGIN_CITY',
    'LATITUDE': 'ORIGIN_LATITUDE', 'LONGITUDE': 'ORIGIN_LONGITUDE'
})
dest_lkp = airports_df.rename(columns={
    'IATA_CODE': 'DESTINATION_AIRPORT', 'AIRPORT': 'DEST_AIRPORT_NAME', 'CITY': 'DEST_CITY',
    'LATITUDE': 'DEST_LATITUDE', 'LONGITUDE': 'DEST_LONGITUDE'
})

df_merged = df_merged.merge(origin_lkp[['ORIGIN_AIRPORT', 'ORIGIN_AIRPORT_NAME', 'ORIGIN_CITY', 'ORIGIN_LATITUDE', 'ORIGIN_LONGITUDE']], on='ORIGIN_AIRPORT', how='left')
df_merged = df_merged.merge(dest_lkp[['DESTINATION_AIRPORT', 'DEST_AIRPORT_NAME', 'DEST_CITY', 'DEST_LATITUDE', 'DEST_LONGITUDE']], on='DESTINATION_AIRPORT', how='left')
df_merged['LATITUDE'] = df_merged['ORIGIN_LATITUDE']
df_merged['LONGITUDE'] = df_merged['ORIGIN_LONGITUDE']
print(df_merged.shape)
display(df_merged.head(2))

## 2- Data Ingestion & Memory Optimization

In [ ]:
df_merged.info()

In [ ]:
# Remove duplicates (although there is no duplicates)
df_merged=df_merged.drop_duplicates().reset_index(drop=True)

In [ ]:
# Data types optimization to save memory
df_merged[['CANCELLED','DIVERTED']].astype('int8')

for col in ['MONTH', 'DAY', 'DAY_OF_WEEK']:
    df_merged[col] = df_merged[col].astype('int8')
df_merged['YEAR'] = df_merged['YEAR'].astype('int16')

object_cols = df_merged.select_dtypes(include=['object']).columns
for col in object_cols:
    df_merged[col] = df_merged[col].astype('category')

float_cols=df_merged.select_dtypes(include=['float64']).columns
for col in float_cols:
    df_merged[col]=df_merged[col].astype('float32')



In [ ]:
num_cols = df_merged.select_dtypes(include= 'number').columns
for col in num_cols:

    print(col)
    print(df_merged[col].nunique())
    print(df_merged[col].unique())
    print('-' * 100)

In [ ]:
cat_cols = df_merged.select_dtypes(include= 'category').columns
for col in cat_cols:

    print(col)
    print(df_merged[col].nunique())
    print(df_merged[col].unique())
    print('-' * 100)

In [ ]:
df_merged.info()

## 3- Defensive Data Cleaning & Business Logic
Handling Missing Values (NaNs) logically:
- Successful flights with no cancellation reason are explicitly marked as 'N'.
- Cancelled flights physically have 0 minutes of delay, air time, and taxi time.

notes to be applied : 
- as the data only in 2015 and the year only have one unique value so we have to delete it to save memory
- we observe that we had only 14 airlines for the 5.8M flights over 2015
- from the unique values of tail numbers we can observe that we had only 4,857 plane
- we had 322 for takeoff and 322 for landing distributed into 308 city



In [ ]:
(df_merged.isna().sum() / len(df_merged) * 100).round(2).sort_values(ascending=False)

In [ ]:
df_merged['AIRLINE'].head()

In [ ]:
# ==============================================================================
# PHASE 1: DIMENSIONALITY REDUCTION & DROPPING NULLS IN CRITICAL COLUMNS
# ==============================================================================
# Dropping redundant keys (since we extracted text names) and zero-variance/macro-level columns.

cols_to_drop = ['YEAR','AIRLINE','ORIGIN_AIRPORT', 'DESTINATION_AIRPORT','LATITUDE','LONGITUDE']
df_merged.drop(columns=cols_to_drop, inplace=True)
df_merged = df_merged.dropna(subset=['TAIL_NUMBER'])

In [ ]:
# ==============================================================================
# PHASE 2: HANDLING CANCELLATIONS & CATEGORICAL LOGIC
# ==============================================================================
# 'N': Not Cancelled (successful flights missing a reason).
# 'U': Unknown (cancelled flights missing the specific reason).

df_merged['CANCELLATION_REASON'] = np.where(
    (df_merged['CANCELLED'] == 0) & (df_merged['CANCELLATION_REASON'].isnull()),
    'N',
    df_merged['CANCELLATION_REASON']
)
df_merged['CANCELLATION_REASON'] = df_merged['CANCELLATION_REASON'].fillna('U')

# ==============================================================================
# PHASE 3: HANDLING OPERATIONAL DURATIONS & DELAYS (NUMERICAL)
# ==============================================================================
operational_cols = ['ARRIVAL_DELAY', 'AIR_TIME', 'ELAPSED_TIME', 'WHEELS_ON', 
                    'ARRIVAL_TIME', 'TAXI_IN', 'TAXI_OUT', 'WHEELS_OFF', 
                    'DEPARTURE_DELAY', 'DEPARTURE_TIME']

delay_cols = ['AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 
              'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY']

df_merged[operational_cols] = df_merged[operational_cols].fillna(0)
df_merged[delay_cols] = df_merged[delay_cols].fillna(0)

# ==============================================================================
# PHASE 4: HANDLING THE FAA ANOMALY (MISSING AIRPORT DATA)
# ==============================================================================
unmapped_cols = ['ORIGIN_AIRPORT_NAME', 'ORIGIN_CITY', 'DEST_AIRPORT_NAME', 'DEST_CITY']
for col in unmapped_cols:
    df_merged[col] = df_merged[col].cat.add_categories(['Unknown']).fillna('Unknown')
# WHY WE ARE KEEPING NaNs IN COORDINATE COLUMNS (ORIGIN_LAT/LONG, DEST_LAT/LONG):
# 1. Geospatial Integrity: Filling missing coordinates with 0 would incorrectly plot these 
#    flights at 'Null Island' (0° Lat, 0° Long) in the Atlantic Ocean, ruining map visualizations.
# 2. Data Preservation: Dropping these rows would cost us ~8.45% of our dataset. We want to 
#    keep these rows because their operational data (delays, airlines) is still highly valuable.
# 3. Tool Compatibility: Modern BI tools (Power BI, Tableau) and Python plotting libraries 
#    automatically ignore NaNs when rendering maps, making it perfectly safe to leave them as-is.

In [ ]:
# investigating when the unkown Airports occurred
unkonwn_flights=df_merged[df_merged['ORIGIN_AIRPORT_NAME']=='Unknown']

missing_by_month=unkonwn_flights.groupby('MONTH').size().reset_index(name='Missing_Airports_Count')

fig_october = px.bar(missing_by_month, x='MONTH', y='Missing_Airports_Count', 
                     title='Anomaly Detection: Missing Airport Names by Month (2015)',
                     labels={'MONTH': 'Month of the Year', 'Missing_Airports_Count': 'Number of Unmapped Airports'},
                     text='Missing_Airports_Count',
                     color='Missing_Airports_Count', color_continuous_scale='Reds')

fig_october.update_xaxes(tickmode='linear', dtick=1)
fig_october.update_traces(textposition='outside')
fig_october.show()
# WHY WE ARE DOING THIS:
# The 8.35% missing values correspond to the flights in October where the FAA switched from IATA letters to numeric IDs.
# Since we dropped the redundant code columns for memory optimization, we will label these as 'Unknown' 
# to keep the dataset clean and ready for visualization without dropping valuable rows.


In [ ]:
# saving the dataframe to a CSV file
df_merged.to_csv('Cleaned_Airlines_Crisis.csv', index=False)

In [ ]:
Cleaned_df=pd.read_csv(r'Airlines_db\Cleaned_Airlines_Crisis.csv')
Cleaned_df.head()

In [ ]:
# Compressing the cleaned data to be able to load in github for streamlit app

df = pd.read_csv(r'Airlines_db\Cleaned_Airlines_Crisis.csv')

crisis_mask = (df['CANCELLED'] == 1) | (df['DIVERTED'] == 1) | (df['WEATHER_DELAY'] > 0)
df_crisis = df[crisis_mask]
df_normal = df[~crisis_mask].sample(frac=0.05, random_state=42)
df_slim = pd.concat([df_crisis, df_normal]).reset_index(drop=True)

essential_cols = [
    'MONTH', 'AIRLINE_NAME', 'ORIGIN_AIRPORT_NAME', 'DEST_AIRPORT_NAME', 'DEST_CITY',
    'ORIGIN_LATITUDE', 'ORIGIN_LONGITUDE', 'DEST_LATITUDE', 'DEST_LONGITUDE',
    'CANCELLED', 'DIVERTED', 'TAXI_OUT', 'WEATHER_DELAY', 'TAXI_IN', 'CANCELLATION_REASON'
]
available_cols = [c for c in essential_cols if c in df_slim.columns]
df_slim = df_slim[available_cols]

df_slim.to_parquet('Airlines_Dashboard_Slim.parquet')
print("✅ Slim file is ready for the Dashboard.")

## Visualization

### Chapter 1: Advanced Financial Engineering(The Bleed)

In [ ]:
# ==============================================================================
# CHAPTER 1: FINANCIAL ENGINEERING & IMPACT (REVENUE VS. PENALTIES)
# ==============================================================================

# 1. Calculate Estimated Base Revenue (Assumption: $25 per mile for successful flights)
Cleaned_df['Estimated_Revenue'] = np.where(Cleaned_df['CANCELLED'] == 0, Cleaned_df['DISTANCE'] * 25, 0)

# 2. Define Airline Business Tiers for Dynamic Penalties
premium_airlines = ['American Airlines Inc.', 'Delta Air Lines Inc.', 'United Air Lines Inc.', 'US Airways Inc.', 'Alaska Airlines Inc.']
budget_airlines = ['Spirit Air Lines', 'Frontier Airlines Inc.']

conditions = [
    (Cleaned_df['CANCELLED'] == 1) & (Cleaned_df['AIRLINE_NAME'].isin(premium_airlines)),
    (Cleaned_df['CANCELLED'] == 1) & (Cleaned_df['AIRLINE_NAME'].isin(budget_airlines)),
    (Cleaned_df['CANCELLED'] == 1) # Standard/Regional
]
choices = [75000, 25000, 50000]
Cleaned_df['Cancellation_Penalty'] = np.select(conditions, choices, default=0)

# 3. Aggregate Financials per Airline
airline_finance = Cleaned_df.groupby('AIRLINE_NAME').agg(
    Total_Revenue=('Estimated_Revenue', 'sum'),
    Total_Penalty=("Cancellation_Penalty", 'sum')
).reset_index()

airline_finance['loss_percentage'] = (airline_finance['Total_Penalty'] / airline_finance['Total_Revenue']).round(4) * 100

# ---------------------------------------------------------
# VISUALIZATIONS: FINANCIAL HEALTH
# ---------------------------------------------------------
# Chart 1.1: Top Revenues
top_revenue = airline_finance.sort_values(by='Total_Revenue', ascending=False).head(10)
fig_rev = px.bar(top_revenue, x='Total_Revenue', y='AIRLINE_NAME', orientation='h',
                 title='<b>Top 10 Airlines by Estimated Revenue (Successful Flights)</b>',
                 labels={'Total_Revenue':'Total Revenue (USD)','AIRLINE_NAME':'Airline'},
                 color='Total_Revenue', color_continuous_scale='Greens')
fig_rev.update_layout(yaxis={'categoryorder':'total ascending'})
fig_rev.show()

# Chart 1.2: Top Bleeders (Absolute Loss)
top_losses = airline_finance.sort_values(by='Total_Penalty', ascending=False).head(10)
fig_loss = px.bar(top_losses, x='Total_Penalty', y='AIRLINE_NAME', orientation='h',
                  title='<b>Top 10 Airlines by Financial Bleed (Cancellation Penalties)</b>',
                  labels={'Total_Penalty': 'Total Penalty (USD)', 'AIRLINE_NAME': 'Airline'},
                  color='Total_Penalty', color_continuous_scale='Reds')
fig_loss.update_layout(yaxis={'categoryorder':'total ascending'})
fig_loss.show()

# Chart 1.3: The Real Impact (% Loss)
airline_finance = airline_finance.sort_values(by='loss_percentage', ascending=False)
fig_reputation = px.bar(airline_finance, x='loss_percentage', y='AIRLINE_NAME', orientation='h',
                        title='<b>The Real Impact: Which Airlines Suffered the Most?</b><br><sup>(Penalty as a % of Total Estimated Revenue)</sup>',
                        labels={'loss_percentage': '% of Revenue Lost', 'AIRLINE_NAME': 'Airline'},
                        color='loss_percentage', color_continuous_scale='Reds')
fig_reputation.update_layout(yaxis={'categoryorder':'total ascending'})
fig_reputation.show()

In [ ]:
import holidays

In [ ]:
temp_dates=pd.DataFrame({'year':2015,'month':Cleaned_df['MONTH'],'day':Cleaned_df['DAY']})
Cleaned_df['DATE']=pd.to_datetime(temp_dates)

us_holidays=holidays.US(years=2015)
Cleaned_df['Is_Holiday']=Cleaned_df['DATE'].apply(lambda x:x in us_holidays)
Cleaned_df['Is_Weekend']=Cleaned_df['DATE'].dt.dayofweek.isin([5,6])# cause weekends in us is sat and sun


In [ ]:
corr_cols = ['DEPARTURE_DELAY', 'ARRIVAL_DELAY', 'WEATHER_DELAY', 'Is_Holiday', 'Is_Weekend', 'DISTANCE']
corr_matrix = Cleaned_df[corr_cols].corr()

fig_corr = px.imshow(
    corr_matrix,
    text_auto='.2f',
    aspect="auto",
    color_continuous_scale='RdBu_r', 
    title='<b>Feature Correlation Analysis</b><br><sup>Understanding the hidden links between Holidays and Delays</sup>'
)
fig_corr.show()

In [ ]:
holiday_impact = Cleaned_df.groupby('Is_Holiday')['DEPARTURE_DELAY'].mean().reset_index()
print(holiday_impact)

weekend_impact = Cleaned_df.groupby('Is_Weekend')['DEPARTURE_DELAY'].mean().reset_index()
print(weekend_impact)


### Chapter 2: Root Causes & Outliers (The Black Swans)

In [ ]:
# ==============================================================================
# CHAPTER 2: ROOT CAUSE ANALYSIS & BLACK SWAN EVENTS
# ==============================================================================

# Chart 2.1: Root Causes of Flight Cancellations (Pie Chart)
reason_mapping = {
    'N': "Not Cancelled", 'A': "Airline / Carrier Issue", 
    'B': "Weather Conditions", 'C': "National Air System", 'D': "Security Reasons"
}
cancelled_flights = Cleaned_df[Cleaned_df['CANCELLED'] == 1].copy()
cancelled_flights['Reason_Text'] = cancelled_flights['CANCELLATION_REASON'].map(reason_mapping)

reason_counts = cancelled_flights['Reason_Text'].value_counts().reset_index()
reason_counts.columns = ['Cancellation_Reason', 'Count']

fig_reasons = px.pie(reason_counts, values='Count', names='Cancellation_Reason',
                     title='<b>Root Causes of Flight Cancellations</b><br><sup>What forces airlines to bleed money?</sup>',
                     color_discrete_sequence=px.colors.qualitative.Set2, hole=0.4)
fig_reasons.update_traces(textposition='inside', textinfo='percent+label')
fig_reasons.show()

# Chart 2.2: Extreme Delays Outlier Analysis
# Note: In Crisis Management, outliers ARE the crisis. We do not drop them.
df_sample = Cleaned_df.sample(frac=0.3, random_state=42)
fig_outliers = px.box(df_sample, x='AIRLINE_NAME', y='DEPARTURE_DELAY',
                      title='<b>Outlier Analysis: The Reality of Extreme Delays (Black Swan Events)</b>',
                      labels={'DEPARTURE_DELAY': 'Departure Delay (Minutes)', 'AIRLINE_NAME': 'Airline'},
                      color='AIRLINE_NAME')
fig_outliers.update_layout(showlegend=False)
fig_outliers.show()

### Chapter 3: Fleet Operations & Maintenance (New Insights)

In [ ]:
# ==============================================================================
# CHAPTER 3: FLEET OPERATIONS & PREDICTIVE MAINTENANCE
# ==============================================================================

# Chart 3.1: Fleet Utilization (Efficiency)
utilization_df = Cleaned_df.groupby('AIRLINE_NAME').agg(
    Total_Flights=('FLIGHT_NUMBER', 'count'),           
    Fleet_Size=('TAIL_NUMBER', 'nunique')               
).reset_index()
utilization_df['Flights_Per_Aircraft'] = utilization_df['Total_Flights'] / utilization_df['Fleet_Size']
utilization_df = utilization_df.sort_values('Flights_Per_Aircraft', ascending=False)

fig_utilization = px.bar(utilization_df, x='AIRLINE_NAME', y='Flights_Per_Aircraft',
                         title='<b>Fleet Utilization: Average Flights Per Aircraft</b><br><sup>Higher indicates efficient use of expensive assets</sup>',
                         labels={'AIRLINE_NAME': 'Airline', 'Flights_Per_Aircraft': 'Avg Flights per Aircraft'},
                         color='Flights_Per_Aircraft', color_continuous_scale='Teal', text_auto='.0f')
fig_utilization.update_layout(xaxis_tickangle=-45)
fig_utilization.show()

# Chart 3.2: Predictive Maintenance Alert
bad_planes_df = Cleaned_df.groupby(['TAIL_NUMBER', 'AIRLINE_NAME']).agg(
    Total_Late_Delay_Mins=('LATE_AIRCRAFT_DELAY', 'sum')
).reset_index()
top_bad_planes = bad_planes_df.sort_values('Total_Late_Delay_Mins', ascending=False).head(15)

fig_maintenance = px.bar(top_bad_planes, x='TAIL_NUMBER', y='Total_Late_Delay_Mins', color='AIRLINE_NAME',
                         title='<b>Predictive Maintenance Alert: Top 15 Aircraft Causing Delays</b><br><sup>These specific planes need immediate mechanical inspection</sup>',
                         labels={'TAIL_NUMBER': 'Aircraft Tail Number', 'Total_Late_Delay_Mins': 'Total Late Delay (Mins)'},
                         text_auto='.0f')
fig_maintenance.show()

### Chapter  4: The Winners (Diversions & Alternative Airports)

In [ ]:
# ==============================================================================
# CHAPTER 4: THE WINNERS (DIVERSIONS & ALTERNATIVE AIRPORTS)
# ==============================================================================

# 1. Alternative Revenue Calculation for Diverted Flights
diverted_flights = Cleaned_df[Cleaned_df['DIVERTED'] == 1].copy()
# Formula: $1500 Base Emergency Fee + ($100 per minute of TAXI_IN congestion)
diverted_flights['Alternative_Revenue'] = 1500 + (diverted_flights['TAXI_IN'] * 100)

airport_gains = diverted_flights.groupby('DEST_AIRPORT_NAME').agg(
    Total_Diverted=('DIVERTED', 'count'),
    Total_Revenue=('Alternative_Revenue', 'sum')
).reset_index()
airport_gains = airport_gains[airport_gains['DEST_AIRPORT_NAME'] != 'Unknown']

# Chart 4.1: Top 10 Crisis Hubs by Revenue
top_airports = airport_gains.sort_values(by='Total_Revenue', ascending=False).head(10)
fig_hubs = px.bar(top_airports, x="Total_Revenue", y='DEST_AIRPORT_NAME', orientation='h',
                  title='<b>Top 10 Crisis Hubs by Unplanned Revenue (2026 Pricing)</b><br><sup>Airports profiting from emergency diversions</sup>',
                  labels={'Total_Revenue': 'Total Unplanned Revenue ($)', 'DEST_AIRPORT_NAME': 'Airport'},
                  color='Total_Diverted', color_continuous_scale='Greens')
fig_hubs.update_layout(yaxis={'categoryorder':'total ascending'})
fig_hubs.show()

# Chart 4.2: Operational Bottlenecks (Ground Congestion)
Cleaned_df['Flight_Status'] = np.where(Cleaned_df['DIVERTED'] == 1, 'Diverted (Emergency)', "Normal Arrival")
top_5_hubs = top_airports['DEST_AIRPORT_NAME'].head(5).tolist()
bottleneck_df = Cleaned_df[Cleaned_df['DEST_AIRPORT_NAME'].isin(top_5_hubs)].copy()

fig_bottleneck = px.box(bottleneck_df, x='DEST_AIRPORT_NAME', y='TAXI_IN', color='Flight_Status',
                        title='<b>Operational Bottlenecks: Ground Congestion (Taxi-In Times)</b><br><sup>Do emergency flights spend more time on the tarmac?</sup>',
                        labels={'TAXI_IN': 'Taxi-In Time (Minutes)', 'DEST_AIRPORT_NAME': 'Airport'},
                        color_discrete_map={'Normal Arrival': '#1f77b4', 'Diverted (Emergency)': '#d62728'})
fig_bottleneck.update_yaxes(range=[0, 60]) # Capping for visibility
fig_bottleneck.show()

In [ ]:
month_map = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun', 
             7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'}

weather_impact = Cleaned_df.groupby('MONTH').agg(
    Avg_Weather_Delay=('WEATHER_DELAY', 'mean'),
    Avg_Taxi_Out=('TAXI_OUT', 'mean')
).reset_index()

weather_impact['Month_Name'] = weather_impact['MONTH'].map(month_map)

# 2. (Multi-Axis Chart)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig_weather = make_subplots(specs=[[{"secondary_y": True}]])

fig_weather.add_trace(
    go.Scatter(x=weather_impact['Month_Name'], y=weather_impact['Avg_Weather_Delay'], 
               name="Avg Weather Delay (Mins)", line=dict(color='blue', width=4)),
    secondary_y=False,
)

fig_weather.add_trace(
    go.Scatter(x=weather_impact['Month_Name'], y=weather_impact['Avg_Taxi_Out'], 
               name="Avg TAXI_OUT (Mins)", line=dict(color='red', width=4, dash='dot')),
    secondary_y=True,
)

fig_weather.update_layout(
    title='<b>The Winter Domino Effect: Weather vs. Taxi-Out</b><br><sup>Proof that snow in Q1 causes ground paralysis</sup>',
    xaxis_title='Month',
    hovermode='x unified',
    template='plotly_white'
)

fig_weather.update_yaxes(title_text="Weather Delay (Mins)", secondary_y=False)
fig_weather.update_yaxes(title_text="Taxi-Out Time (Mins)", secondary_y=True)

fig_weather.show()

In [ ]:
# ==============================================================================
# PHASE 4.3: GEOGRAPHICAL WEATHER IMPACT (US MAP)
# ==============================================================================

weather_geo = Cleaned_df.groupby(['ORIGIN_AIRPORT_NAME', 'ORIGIN_LATITUDE', 'ORIGIN_LONGITUDE']).agg(
    Avg_Weather_Delay=('WEATHER_DELAY', 'mean'),
    Total_Flights=('FLIGHT_NUMBER', 'count')
).reset_index()

weather_geo = weather_geo.dropna(subset=['ORIGIN_LATITUDE', 'ORIGIN_LONGITUDE'])
weather_geo = weather_geo[weather_geo['Avg_Weather_Delay'] > 0]

# 3. Plotly Scatter Geo
fig_weather_map = px.scatter_geo(
    weather_geo,
    lat='ORIGIN_LATITUDE',     
    lon='ORIGIN_LONGITUDE',     
    color='Avg_Weather_Delay', 
    size='Avg_Weather_Delay',  
    hover_name='ORIGIN_AIRPORT_NAME',
    scope='usa',               
    title='<b>The Geography of Weather: Average Weather Delay by Airport</b><br><sup>Does location determine the risk of climate-driven paralysis?</sup>',
    labels={'Avg_Weather_Delay': 'Avg Delay (Mins)'},
    color_continuous_scale='YlOrRd', 
    template='plotly_white'
)

# تحسين شكل الخريطة
fig_weather_map.update_geos(
    resolution=50,
    showcoastlines=True, coastlinecolor="RebeccaPurple",
    showland=True, landcolor="LightGreen",
    showocean=True, oceancolor="LightBlue",
    showlakes=True, lakecolor="Blue",
    showcountries=True
)

fig_weather_map.show()

In [ ]:
# ==============================================================================
# PHASE 4.4: REFINED WINTER TAXI-OUT MAP (High Contrast & Large Scale)
# ==============================================================================

winter_months = [1, 2, 12]
winter_df = Cleaned_df[Cleaned_df['MONTH'].isin(winter_months)].copy()

taxi_out_geo = winter_df.groupby(['ORIGIN_AIRPORT_NAME', 'ORIGIN_LATITUDE', 'ORIGIN_LONGITUDE']).agg(
    Avg_Taxi_Out=('TAXI_OUT', 'mean')
).reset_index()

taxi_out_geo = taxi_out_geo.dropna(subset=['ORIGIN_LATITUDE', 'ORIGIN_LONGITUDE'])

fig_winter_taxi = px.scatter_geo(
    taxi_out_geo,
    lat='ORIGIN_LATITUDE',
    lon='ORIGIN_LONGITUDE',
    color='Avg_Taxi_Out',
    size='Avg_Taxi_Out',
    hover_name='ORIGIN_AIRPORT_NAME',
    scope='usa',
    color_continuous_scale='IceFire', 
    title='<b>Winter Operational Analysis: Ground Congestion (TAXI_OUT)</b><br><sup>Visualizing the Northern "Snow Belt" Delay Patterns</sup>',
    labels={'Avg_Taxi_Out': 'Avg Taxi-Out (Mins)'},
    template='plotly_white',
    height=800 
)

fig_winter_taxi.update_traces(
    marker=dict(line=dict(width=0.5, color='white')), 
    selector=dict(mode='markers')
)

fig_winter_taxi.update_geos(
    projection_type="albers usa",    
    showland=True, landcolor="rgb(240, 240, 240)",
    showlakes=True, lakecolor="white",
    showrivers=False,
    subunitcolor="rgb(200, 200, 200)"
)

fig_winter_taxi.show()

### Chapter 5: Crisis Mapping (Timing & Geography)
**Business Objective:** Identify spatial and temporal patterns to proactively predict, plan for, and mitigate future aviation crises.


In [ ]:
# ==============================================================================
# CHAPTER 5: CRISIS MAPPING (TIMING & GEOGRAPHY)
# ==============================================================================
import plotly.express as px

severe_disruptions = Cleaned_df[(Cleaned_df['CANCELLED'] == 1) | (Cleaned_df['DIVERTED'] == 1)].copy()

severe_disruptions = severe_disruptions[severe_disruptions['ORIGIN_AIRPORT_NAME'] != 'Unknown']
top_10_risk_airports = severe_disruptions['ORIGIN_AIRPORT_NAME'].value_counts().nlargest(10).index

crisis_data = severe_disruptions[severe_disruptions['ORIGIN_AIRPORT_NAME'].isin(top_10_risk_airports)]

crisis_map = crisis_data.groupby(['MONTH', 'ORIGIN_AIRPORT_NAME']).size().reset_index(name='Disruption_Count')

month_map = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun', 
             7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'}
crisis_map['Month_Name'] = crisis_map['MONTH'].map(month_map)

# 4.Scatter Plot (Bubble Chart)
fig_crisis = px.scatter(
    crisis_map,
    x='Month_Name',
    y='Disruption_Count',
    size='Disruption_Count',     
    color='ORIGIN_AIRPORT_NAME',  
    hover_name='ORIGIN_AIRPORT_NAME',
    title='<b>Crisis Mapping: Ground Zero Airports & Temporal Peaks</b><br><sup>When and where do severe disruptions (Cancellations & Diversions) peak?</sup>',
    labels={
        'Month_Name': 'Month of the Year',
        'Disruption_Count': 'Number of Severe Disruptions',
        'ORIGIN_AIRPORT_NAME': 'Origin Airport'
    },
    size_max=35, 
    template='plotly_white'
)


fig_crisis.update_xaxes(categoryorder='array', categoryarray=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
fig_crisis.update_traces(marker=dict(line=dict(width=1, color='DarkSlateGrey')), opacity=0.8)

fig_crisis.show()